In [12]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.utils import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling, Input
from keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping
from keras.optimizers import SGD

import keras_cv

mixed_precision.set_global_policy("mixed_float16")

In [13]:
# directory where images are located
data_dir = "../data/original"
image_size = (32, 32)
batch_size = 32

file_paths = []
labels = []

# the name of each folder corresponds to the label of the images within
class_names = os.listdir(data_dir)
class_to_index = {name: i for i, name in enumerate(class_names)}

# for each folder, we'll save the path of each file with its correspondent label
for class_name in class_names:
    class_dir = f"{data_dir}/{class_name}"
    images = sorted(os.listdir(class_dir))
    for img_path in images:
        file_paths.append(class_dir + "/" + str(img_path))
        labels.append(class_to_index[class_name])

# now we have all the paths and labels together
file_paths = np.array(file_paths)

# bad_files = []

# for path in file_paths:
#     try:
#         if not check_jpeg(path):
#             bad_files.append(path)
#     except:
#         bad_files.append(path)

# print("invalid files:" +  str(len(bad_files)))

# todo: include if

# fix bad_files (already done)
# for path in bad_files:
#     img = Image.open(path)
#     new_path = path.rsplit(".", 1)[0] + ".jpg"
#     img.convert("RGB").save(new_path, "JPEG")

In [14]:
labels = np.array(labels)
dimension = 3

X_train, X_test, y_train, y_test = train_test_split(
    file_paths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

cv_train_set = tf.data.Dataset.from_tensor_slices((X_train, y_train))
cv_train_set = cv_train_set.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
cv_train_set = cv_train_set.shuffle(int(len(file_paths))).batch(batch_size).prefetch(tf.data.AUTOTUNE)

cv_test_set = tf.data.Dataset.from_tensor_slices((X_test, y_test))
cv_test_set = cv_test_set.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
cv_test_set = cv_test_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [4]:
class_weights = dict(enumerate(class_weights))

In [5]:
def build_baseline_model(n_filters = 64, n_neurons = 64, learning_rate = 1e-2, momentum = 0.0, 
                      kernel_size = 3, pool_size = 2, sparse = False, **kwargs):
    '''Builds the first CNN architecture. This model is intended to be very simple, to start understanding how well it performs over the dataset
        and to start developing more complex models. It uses basic concepts of CNNs: a convolution layer followed by a pooling layer, then a dense layer with a final
        softmax given output.
        
    Args:
        - n_filters (int): number of filters to be used on convolution layers (Conv2D). Default set to 64.
        - n_neurons (int): number of neurons to be used on dense layers (Dense). Default set to 64.
        - learning_rate (float): indicates a which rate the gradient moves. Default set to 1e-2.
        - momentum (float): proportion regarding how much of past gradients affect the current gradient step. Default set to 0.0.
        - kernel_size (int): indicates the size of the squared sliding window over the feature map. Default set to 3.
        - pool_size (int): indicated the size of the downsample pooling window. Default set to 2.
        - sparse (bool): sets if labels should be considered as one-hot (False) or integers (True). Default set to True.
        - other parameters such as activation function (ReLU), padding (same) and activation (softmax) were kept as default throught all architectures.
    Returns:
        - model: fully compiled model with all respective features.
    
    '''
    
    normalization_layer = Rescaling(1./255)
    
    model = Sequential([
    Input(shape = (image_size[0], image_size[1], dimension)),
    normalization_layer,
    Conv2D(n_filters, kernel_size, activation = "relu", padding = "same"),
    MaxPooling2D(pool_size),
    Flatten(),
    Dense(n_neurons, activation="relu"),
    Dense(10, activation = "softmax")
                    ])
    
    # for the first model only SGD optimizer is used.
    optimizer = SGD(learning_rate=learning_rate, momentum=momentum)
    
    loss = "categorical_crossentropy"
    if sparse:
        loss = "sparse_categorical_crossentropy"
    model.compile(loss = loss,
        optimizer = optimizer,
        metrics = ["accuracy"])
    
    return model

In [6]:
# X_train_array, y_train_array = pass_batchs_to_arrays(cv_train_set, pass_dummy=True)
# X_test_array, y_test_array = pass_batchs_to_arrays(cv_test_set, pass_dummy=True)

# param_grid = {"learning_rate": [1e-2, 1e-3],
#               "momentum": [0.0,0.9],
#               "kernel_size": [2, 3],
#               "n_filters": [32, 64],
#               "n_neurons": [32, 64]}

param_grid = {"learning_rate": [1e-2],
              "momentum": [0.0],
              "kernel_size": [2],
              "n_filters": [32],
              "n_neurons": [32]}

# X_train, X_test, y_train, y_test
best_params, best_accuracy, mean_scores  = CNN_GridSearchCV(X_train, y_train, param_grid, build_baseline_model, 42, 5, n_jobs = -1)

In [10]:
best_params

{'learning_rate': 0.01,
 'momentum': 0.0,
 'kernel_size': 2,
 'n_filters': 32,
 'n_neurons': 32}

In [11]:
mean_scores

[(np.float64(1.4841594219207763), np.float64(0.4841727912425995))]

In [15]:
which_arch = "baseline"

run_id, run_logdir = get_run_logdir(which_arch, "1")

# different callbacks to: save the best model over validation set, save logs of loss and accuracy for later visualization and early stopping.
model_checkpoint = ModelCheckpoint(run_logdir +"\\baseline.keras", save_best_only=True)
tensorboard_cb = TensorBoard(run_logdir)
early_stopping = EarlyStopping(patience = 5, restore_best_weights=True)

# compile the model
model = build_baseline_model(**best_params, sparse= False)

# train
# we use a higher number of epochs since early stopping should stop since it reaches a no-improvement point.
model.fit(cv_train_set, epochs=30, callbacks= [tensorboard_cb,model_checkpoint, early_stopping])

table_from_history(model.history.history, run_id).to_csv("curves_data/"+which_arch+"/"+run_id+".csv")

# with the best parameters achieved, we train the first model and evaluate for the test error
save_params(run_logdir, "best_params_baseline_model.pkl",best_params)

Epoch 1/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.1772 - loss: 2.2140
Epoch 2/30


c:\Users\faarc\anaconda3\envs\deepl\Lib\site-packages\keras\src\callbacks\model_checkpoint.py:302: UserWarning: Can save best model only with val_loss available.
  if self._should_save_model(epoch, batch, logs, filepath):
c:\Users\faarc\anaconda3\envs\deepl\Lib\site-packages\keras\src\callbacks\early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.2730 - loss: 2.0244
Epoch 3/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.3350 - loss: 1.8673
Epoch 4/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.3715 - loss: 1.7825
Epoch 5/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.4015 - loss: 1.7089
Epoch 6/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.4254 - loss: 1.6327
Epoch 7/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.4430 - loss: 1.5971
Epoch 8/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.4653 - loss: 1.5397
Epoch 9/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.4866 - loss: 1.4829
Epoch 10/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - accuracy: 0.4964 - loss: 1.4586
Epoch 11/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.5200 - loss: 1.4129
Epoch 12/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.5262 - loss: 1.3872
Epoch 13/30
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms

In [16]:
model.evaluate(cv_test_set)

84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6085 - loss: 1.2740


[1.2935632467269897, 0.5846441984176636]

In [17]:
y_pred = np.argmax(model.predict(cv_test_set), axis = 1)

84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [19]:
# cm = compute_classification_metrics(y_test_array, y_pred, class_names)